# Trabalho Prático de IA - Notebook base de coleta de dados do ArXiv

**Disciplina:** Inteligência Artificial - PPGCC - FACOM/UFMS  
**Professor:** Bruno M. Nogueira  
**Semestre:** 2026/1

Este notebook é um **ponto de partida** para o trabalho prático. Ele mostra como **coletar artigos do ArXiv** filtrando por categorias e palavras-chave relacionadas ao seu tema de pesquisa, normalizar os metadados e salvar uma coleção em `JSONL`.

Você é **livre (e encorajado)** para modificar este código: trocar a API, ajustar os filtros, adicionar campos, etc. O notebook serve apenas para que você não perca tempo na parte mais mecânica do trabalho.

**Atenção:** a etapa de coleta deve ser justificada no relatório (escolha do tema, palavras-chave, categorias, janela temporal). Veja a Seção 3-(a) do enunciado e o exemplo lá fornecido.

> **A API do ArXiv é instável.** Não é incomum ver erros `HTTP 429` ("too many requests") ou `HTTP 503` ("service unavailable"), especialmente em queries longas ou em horários de pico. Este notebook implementa **retry com backoff exponencial e retomada por offset**, e o arquivo de saída preserva o progresso. Se a célula de coleta falhar, basta **rodar a mesma célula novamente** mais tarde, ela continua de onde parou. Não existe "chave de API" do ArXiv: o limite é por IP.

## 1. Instalação de dependências

Vamos usar a biblioteca [`arxiv`](https://pypi.org/project/arxiv/), um *wrapper* sobre a API oficial do ArXiv que já cuida de paginação e *rate limiting* básico. Também usamos `pandas` para inspeção tabular.

In [7]:
# Execute apenas uma vez, descomentando se necessário.
# %pip install arxiv pandas tqdm


   ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
   --------------- ------------------------ 1.6/4.0 MB 10.5 MB/s eta 0:00:01
   ---------------------------------------  3.9/4.0 MB 10.7 MB/s eta 0:00:01
   ---------------------------------------- 4.0/4.0 MB 9.9 MB/s  0:00:00
   ---------------------------------------- 0.0/9.8 MB ? eta -:--:--
   --------- ------------------------------ 2.4/9.8 MB 11.9 MB/s eta 0:00:01
   -------------------- ------------------- 5.0/9.8 MB 11.8 MB/s eta 0:00:01
   ------------------------------ --------- 7.3/9.8 MB 11.8 MB/s eta 0:00:01
   ---------------------------------------  9.7/9.8 MB 11.7 MB/s eta 0:00:01
   ---------------------------------------- 9.8/9.8 MB 11.2 MB/s  0:00:00
   ---------------------------------------- 0.0/12.3 MB ? eta -:--:--
   ------- -------------------------------- 2.4/12.3 MB 11.8 MB/s eta 0:00:01
   ---------------- ----------------------- 5.0/12.3 MB 11.8 MB/s eta 0:00:01
   ----------------------- 


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
import json
import os
import time
from datetime import datetime, timezone
from pathlib import Path

import arxiv
import pandas as pd
from tqdm import tqdm

## 2. Configuração da coleta (você edita aqui!)

Adapte os campos abaixo ao **seu** tema de pesquisa. O exemplo segue o caso do enunciado ("Detecção de fraudes em transações financeiras com aprendizado profundo").

**Dica para reduzir erros 429/503:** use `PAGE_SIZE = 50` (em vez de 100), evite horários de pico (manhã EUA) e mantenha as palavras-chave razoavelmente concisas. Queries muito longas (muitos `OR`) tendem a falhar mais.

In [10]:
# -----------------------------------------------------------------------------
# DEFINA AQUI O ESCOPO DA SUA COLEÇÃO
# -----------------------------------------------------------------------------

# Palavras-chave / frases que descrevem o seu tema. Serão buscadas em
# título e abstract (operador 'all:' do ArXiv).
KEYWORDS = [
    "layoutLM",
    "pre-trained models",
    "document image understand",
    "documentAI",
]

# Categorias do ArXiv a considerar (lista em https://arxiv.org/category_taxonomy).
# Use [] para não restringir por categoria.
CATEGORIES = ["cs.AI", "cs.LG", "cs.MM", "cs.MA", "cs.NE"]

# Janela temporal (ano de submissão). Use None para não restringir.
YEAR_FROM = 2023
YEAR_TO = 2026

# Tamanho-alvo aproximado da coleção (entre 1000 e 5000, conforme enunciado).
TARGET_SIZE = 2000

# Tamanho de cada página retornada pela API.
# 50 costuma ser mais robusto contra HTTP 429/503 do que 100.
PAGE_SIZE = 50

# Caminho de saída da coleção bruta.
OUTPUT_DIR = Path("./data")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RAW_PATH = OUTPUT_DIR / "arxiv_raw.jsonl"
CORPUS_PATH = OUTPUT_DIR / "corpus.jsonl"

print("Configuração carregada.")
print(" - Keywords  :", KEYWORDS)
print(" - Categories:", CATEGORIES)
print(" - Years     :", YEAR_FROM, "->", YEAR_TO)
print(" - Target    :", TARGET_SIZE)
print(" - PageSize  :", PAGE_SIZE)

Configuração carregada.
 - Keywords  : ['layoutLM', 'pre-trained models', 'document image understand', 'documentAI']
 - Categories: ['cs.AI', 'cs.LG', 'cs.MM', 'cs.MA', 'cs.NE']
 - Years     : 2023 -> 2026
 - Target    : 2000
 - PageSize  : 50


## 3. Coleta paginada via API do ArXiv (com retry/backoff)

A função abaixo monta uma *query* combinando palavras-chave (com `OR`) e categorias (com `OR`), e itera sobre os resultados em ordem decrescente de data de submissão.

**Como ela trata erros HTTP 429 (rate limit) e 503 (serviço indisponível):**
- Configura o cliente do `arxiv` com `delay_seconds=5` e `num_retries=8` --- mais paciente que o padrão.
- Encapsula a iteração em um *loop externo* com **backoff exponencial**: 60s → 120s → 240s → 480s → 600s entre tentativas.
- Mantém um *offset* de onde paramos: se cair, retoma a partir do mesmo ponto (sem re-baixar tudo).
- Salva incrementalmente em `arxiv_raw.jsonl` e *deduplica* por `arxiv_id`. **Você pode rodar a célula de coleta várias vezes** --- o progresso é preservado.

**Se mesmo assim não funcionar:** espere 15–30 minutos e tente de novo (o seu IP pode estar temporariamente bloqueado), ou reduza `PAGE_SIZE` para 25 e/ou diminua o número de palavras-chave.

In [11]:
def build_query(keywords, categories):
    """Monta a string de query no formato esperado pela API do ArXiv."""
    kw_part = " OR ".join([f'all:"{k}"' for k in keywords]) if keywords else ""
    cat_part = " OR ".join([f"cat:{c}" for c in categories]) if categories else ""

    parts = [p for p in [f"({kw_part})" if kw_part else "",
                          f"({cat_part})" if cat_part else ""] if p]
    return " AND ".join(parts) if parts else "all:*"


QUERY = build_query(KEYWORDS, CATEGORIES)
print("Query final:\n", QUERY)

Query final:
 (all:"layoutLM" OR all:"pre-trained models" OR all:"document image understand" OR all:"documentAI") AND (cat:cs.AI OR cat:cs.LG OR cat:cs.MM OR cat:cs.MA OR cat:cs.NE)


In [12]:
def already_collected_ids(path: Path) -> set:
    """Lê o arquivo .jsonl (se existir) e retorna os IDs já salvos."""
    if not path.exists():
        return set()
    ids = set()
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            try:
                ids.add(json.loads(line)["arxiv_id"])
            except Exception:
                continue
    return ids


def collect_arxiv(query, target_size, page_size, year_from, year_to,
                   out_path: Path,
                   max_outer_attempts: int = 6,
                   initial_backoff_seconds: int = 60):
    """Coleta artigos do ArXiv, salvando incrementalmente em JSONL.

    Robusto contra erros HTTP 429/503 da API do ArXiv:
    - Retry interno do cliente `arxiv` (delay_seconds=5, num_retries=8).
    - Loop externo com backoff exponencial caso a iteração falhe.
    - Retoma do mesmo offset após cada falha, sem re-baixar páginas inteiras.
    """
    client = arxiv.Client(page_size=page_size, delay_seconds=5, num_retries=8)

    seen = already_collected_ids(out_path)
    print(f"Já coletados anteriormente: {len(seen)} artigos.")

    saved_this_run = 0
    offset = 0          # quantos resultados já consumimos da API
    outer_attempt = 0   # quantas vezes o loop externo tentou

    while len(seen) < target_size and outer_attempt < max_outer_attempts:
        try:
            search = arxiv.Search(
                query=query,
                max_results=target_size * 3,  # 3x para sobrar margem após filtros
                sort_by=arxiv.SortCriterion.SubmittedDate,
                sort_order=arxiv.SortOrder.Descending,
            )

            print(f"\nIniciando/retomando do offset={offset} "
                  f"(salvos={len(seen)}, meta={target_size}).")

            with open(out_path, "a", encoding="utf-8") as f:
                pbar = tqdm(initial=len(seen), total=target_size,
                            desc="coletando")
                for result in client.results(search, offset=offset):
                    offset += 1
                    year = result.published.year if result.published else None
                    if year_from is not None and (year is None or year < year_from):
                        continue
                    if year_to is not None and (year is None or year > year_to):
                        continue

                    arxiv_id = result.get_short_id().split("v")[0]
                    if arxiv_id in seen:
                        continue

                    record = {
                        "arxiv_id": arxiv_id,
                        "title": (result.title or "").strip(),
                        "abstract": (result.summary or "").strip().replace("\n", " "),
                        "authors": [a.name for a in result.authors],
                        "categories": list(result.categories or []),
                        "primary_category": result.primary_category,
                        "published": result.published.isoformat() if result.published else None,
                        "updated": result.updated.isoformat() if result.updated else None,
                        "doi": result.doi,
                        "pdf_url": result.pdf_url,
                        "entry_id": result.entry_id,
                    }
                    f.write(json.dumps(record, ensure_ascii=False) + "\n")
                    f.flush()
                    seen.add(arxiv_id)
                    saved_this_run += 1
                    pbar.update(1)
                    pbar.set_postfix(offset=offset)

                    if len(seen) >= target_size:
                        break
                pbar.close()

            # Se chegou aqui sem exception, terminou normalmente.
            break

        except Exception as e:
            outer_attempt += 1
            wait = min(initial_backoff_seconds * (2 ** (outer_attempt - 1)), 600)
            print(f"\n[aviso] coleta interrompida (tentativa {outer_attempt}/"
                  f"{max_outer_attempts}): {type(e).__name__}: {e}")
            print(f"[aviso] aguardando {wait}s antes de retomar do offset={offset}...")
            for _ in tqdm(range(wait), desc="backoff", leave=False):
                time.sleep(1)

    print(f"\nColeta finalizada. Total acumulado em {out_path}: {len(seen)} artigos.")
    if len(seen) < target_size:
        print(f"[atenção] Não atingiu a meta de {target_size} (parou em {len(seen)}).")
        print( "[atenção] Você pode rodar esta célula novamente para continuar de onde parou.")
    return len(seen)

In [13]:
n = collect_arxiv(
    query=QUERY,
    target_size=TARGET_SIZE,
    page_size=PAGE_SIZE,
    year_from=YEAR_FROM,
    year_to=YEAR_TO,
    out_path=RAW_PATH,
)
n

Já coletados anteriormente: 0 artigos.

Iniciando/retomando do offset=0 (salvos=0, meta=2000).


coletando:   0%|          | 0/2000 [00:00<?, ?it/s]


[aviso] coleta interrompida (tentativa 1/6): ChunkedEncodingError: ("Connection broken: ConnectionResetError(10054, 'Foi forçado o cancelamento de uma conexão existente pelo host remoto', None, 10054, None)", ConnectionResetError(10054, 'Foi forçado o cancelamento de uma conexão existente pelo host remoto', None, 10054, None))
[aviso] aguardando 60s antes de retomar do offset=0...



backoff: 100%|██████████| 60/60 [01:00<00:00,  1.00s/it]
                                                        


Iniciando/retomando do offset=0 (salvos=0, meta=2000).


...


backoff: 100%|██████████| 240/240 [04:00<00:00,  1.00s/it]
                                                          


Iniciando/retomando do offset=300 (salvos=300, meta=2000).



coletando:  15%|█▌        | 300/2000 [05:40<?, ?it/s]

coletando: 100%|██████████| 2000/2000 [07:45<00:00,  3.65it/s, offset=2000]


Coleta finalizada. Total acumulado em data\arxiv_raw.jsonl: 2000 artigos.


2000

### O que fazer se a coleta continuar falhando

1. **Espere e tente de novo.** A API do ArXiv pode temporariamente bloquear o seu IP. Aguarde 15--30 min e rode a célula acima novamente --- ela vai retomar.
2. **Reduza a query.** Menos `OR` $\Rightarrow$ menos custo no servidor. Tente com 2--3 palavras-chave mais discriminativas.
3. **Reduza `PAGE_SIZE`** para 25 (mais requisições, cada uma mais leve).
4. **Aumente o `delay_seconds`** do `arxiv.Client` para 10 ou 15 segundos.
5. **Plano B: use o *bulk access* do ArXiv** (S3 / Kaggle dump) --- baixa em uma vez só, sem rate limit, ao custo de não ter os artigos mais recentes do dia.
6. **Plano C: troque a fonte** para Semantic Scholar, OpenAlex ou DBLP (todas listadas na Seção 5 do enunciado).

## 4. Normalização e deduplicação

A coleta bruta pode conter duplicatas (mesmo `arxiv_id` em diferentes versões) ou registros sem abstract. Geramos uma versão limpa em `corpus.jsonl`.

In [14]:
def load_jsonl(path: Path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f]


raw = load_jsonl(RAW_PATH)
print("Registros brutos:", len(raw))

# Deduplicar por arxiv_id (mantém o mais recente por 'updated').
df = pd.DataFrame(raw)
df["updated_dt"] = pd.to_datetime(df["updated"], errors="coerce")
df = df.sort_values("updated_dt").drop_duplicates("arxiv_id", keep="last")

# Remover registros sem título ou sem abstract.
df = df[df["title"].str.len() > 0]
df = df[df["abstract"].str.len() > 50]  # abstracts muito curtos costumam ser ruído

print("Após deduplicação e limpeza:", len(df))
df.head(3)

Registros brutos: 2000
Após deduplicação e limpeza: 2000


,arxiv_id,title,abstract,authors,categories,primary_category,published,updated,doi,pdf_url,entry_id,updated_dt
1996,2311.07578,A Metacognitive Approach to Out-of-Distributio...,Despite outstanding semantic scene segmentatio...,"[Meghna Gummadi, Cassandra Kent, Karl Schmeckp...","[cs.CV, cs.AI, cs.LG]",cs.CV,2023-10-04T16:37:38+00:00,2023-10-04T16:37:38+00:00,NaN,https://arxiv.org/pdf/2311.07578v1,http://arxiv.org/abs/2311.07578v1,2023-10-04 16:37:38+00:00
1994,2310.03123,Efficient Federated Prompt Tuning for Black-bo...,With the blowout development of pre-trained mo...,"[Zihao Lin, Yan Sun, Yifan Shi, Xueqian Wang, ...","[cs.LG, cs.AI]",cs.LG,2023-10-04T19:30:49+00:00,2023-10-04T19:30:49+00:00,NaN,https://arxiv.org/pdf/2310.03123v1,http://arxiv.org/abs/2310.03123v1,2023-10-04 19:30:49+00:00
1991,2310.03620,PeaTMOSS: Mining Pre-Trained Models in Open-So...,Developing and training deep learning models i...,"[Wenxin Jiang, Jason Jones, Jerin Yasmin, Nich...","[cs.SE, cs.AI]",cs.SE,2023-10-05T15:58:45+00:00,2023-10-05T15:58:45+00:00,NaN,https://arxiv.org/pdf/2310.03620v1,http://arxiv.org/abs/2310.03620v1,2023-10-05 15:58:45+00:00


## 5. Salvamento em `corpus.jsonl`

Este é o arquivo que você vai usar nas próximas etapas (BM25, KNN, etc.).

In [15]:
cols = ["arxiv_id", "title", "abstract", "authors", "categories",
         "primary_category", "published", "doi", "pdf_url"]

with open(CORPUS_PATH, "w", encoding="utf-8") as f:
    for _, row in df[cols].iterrows():
        f.write(json.dumps(row.to_dict(), ensure_ascii=False) + "\n")

print(f"Corpus salvo em: {CORPUS_PATH} ({len(df)} documentos).")

Corpus salvo em: data\corpus.jsonl (2000 documentos).


## 6. Verificação rápida da coleção (*sanity checks*)

Antes de partir para a recuperação, dê uma olhada na distribuição temporal, por categoria, e em alguns exemplos. Erros mais comuns: filtros muito apertados (corpus minúsculo), filtros muito frouxos (corpus contaminado com áreas alheias).

In [16]:
df["year"] = pd.to_datetime(df["published"], errors="coerce").dt.year
print("Distribuição por ano:")
print(df["year"].value_counts().sort_index())
print("\nDistribuição por categoria primária:")
print(df["primary_category"].value_counts().head(10))

Distribuição por ano:
year
2023    200
2024    880
2025    705
2026    215
Name: count, dtype: int64

Distribuição por categoria primária:
primary_category
cs.LG      813
cs.CV      481
cs.CL      237
cs.AI       89
cs.SD       64
eess.AS     37
eess.IV     37
cs.CR       36
cs.SE       27
stat.ML     20
Name: count, dtype: int64


In [17]:
# Exemplos aleatórios para inspeção manual.
for _, row in df.sample(min(3, len(df)), random_state=42).iterrows():
    print("=" * 80)
    print("ID      :", row["arxiv_id"])
    print("Title   :", row["title"])
    print("Cats    :", row["categories"])
    print("Year    :", row["year"])
    print("Abstract:", row["abstract"][:400], "...")

ID      : 2505.19361
Title   : Consistency-based Abductive Reasoning over Perceptual Errors of Multiple Pre-trained Models in Novel Environments
Cats    : ['cs.AI', 'cs.CV', 'cs.LG', 'cs.LO']
Year    : 2025
Abstract: The deployment of pre-trained perception models in novel environments often leads to performance degradation due to distributional shifts. Although recent artificial intelligence approaches for metacognition use logical rules to characterize and filter model errors, improving precision often comes at the cost of reduced recall. This paper addresses the hypothesis that leveraging multiple pre-train ...
ID      : 2406.18550
Title   : Pre-Trained Vision-Language Models as Partial Annotators
Cats    : ['cs.CV', 'cs.AI']
Year    : 2024
Abstract: Pre-trained vision-language models learn massive data to model unified representations of images and natural languages, which can be widely applied to downstream machine learning tasks. In addition to zero-shot inference, in order to be